# 05 — Text Classification (NLP)

**Task 1:** Classify listing description into `cat3_slug` (property type) — Logistic Regression / TF-IDF  
**Task 2:** Classify `user_type` (individual vs. agency) — LSTM

---

## Overview

This notebook covers the full NLP pipeline for Persian real estate listing text.

> **⚠ Google Colab Note:** The original notebook was developed and trained on **Google Colab** (GPU).
> The `from google.colab import drive` cell has been commented out.
> To run locally, ensure your data files are in `../data/` and a GPU is available for LSTM training.

### Sub-stages

| Sub-stage | Source notebook | What it does |
|---|---|---|
| 5a — NLP Cleaning | `4-TEXT CLASSIFICATION/1 - CLEANING.ipynb` | Persian text normalisation with Hazm (tokenisation, stemming, stop-word removal), remove noise |
| 5b — Modelling | `4-TEXT CLASSIFICATION/2 - MODELING.ipynb` | TF-IDF Logistic Regression, Bag-of-Words, Word2Vec embeddings, LSTM classifier |

### Libraries
- **Hazm** (Persian NLP toolkit) — tokenisation, stemming, stopwords
- **Scikit-learn** — TF-IDF vectoriser, Logistic Regression, metrics
- **Gensim** — Word2Vec training
- **TensorFlow / Keras** — LSTM model

### Key Results
- **TF-IDF + Logistic Regression** for `cat3_slug`: Accuracy ~87%, Weighted F1 ~85%
- **LSTM** for `user_type`: High recall for the "agency" class; class imbalance addressed with class weights.

### Input
`../data/divar_tehran_nonnum_filled_final.pkl` (or the raw filtered CSV)

---


## Sub-stage 5a — Persian Text Cleaning (Hazm)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from time import sleep

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(r"C:\Users\kourosh\Desktop\daqiqe Project\1\divar_real_estate_ads.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df = df[["cat3_slug","user_type","description","title"]]

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["not_cleaned"] = df["description"] + " " + df["title"]

In [ ]:
df.head()

In [ ]:
df["not_cleaned"].value_counts

In [ ]:
df["not_cleaned"][2]

In [ ]:
import re

# حذف کاراکترهای غیر الفبایی و اعداد و تبدیل چند فاصله به یکی
df["not_cleaned"] = df["not_cleaned"].str.replace(r"[\s\W\d]", " ", regex=True)
df["not_cleaned"] = df["not_cleaned"].str.replace(r" +", " ", regex=True)

# نتیجه نهایی
text = df["not_cleaned"]


In [ ]:
df.head()

In [ ]:
df["not_cleaned"][2]

In [ ]:
df["not_cleaned"] = df["not_cleaned"].str.strip()

In [ ]:
df["not_cleaned"][2]

In [ ]:
df["not_cleaned"][6]

## در کد پایین خواستم اعداد حفظ شوند و بقیه موارد حذف شوند که اعداد هم معنادار باشد

In [ ]:
df["not_cleaned2"] = df["description"] + " " + df["title"]
df["not_cleaned2"] = df["not_cleaned2"].str.replace(r"[^۰-۹0-9آ-یءچپژگک]", " ", regex=True)
df["not_cleaned2"] = df["not_cleaned2"].str.replace(r" +", " ", regex=True)
df["not_cleaned2"] = df["not_cleaned2"].str.strip()


In [ ]:
df["not_cleaned2"] = df["not_cleaned2"].str.strip()

In [ ]:
df["not_cleaned2"][8]

In [ ]:
df["not_cleaned2"][336]

In [ ]:
df["not_cleaned2"][665]

In [ ]:
df["clean_text"] = df["not_cleaned"]

In [ ]:
df.head()

In [ ]:
df = df[["cat3_slug","user_type","clean_text"]]

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
from hazm import Normalizer

normalizer = Normalizer()

In [ ]:
df = df.dropna(subset=['cat3_slug', 'clean_text']).reset_index(drop=True)

In [ ]:
df.info()

In [ ]:
df["clean_text"] = df["clean_text"].apply(normalizer.normalize)

In [ ]:
df["clean_text"]

In [ ]:
from hazm import word_tokenize
df["tokens"] = df["clean_text"].apply(word_tokenize)

In [ ]:
df["tokens"][50]

In [ ]:
df.info()

In [ ]:
from hazm import Stemmer

stemmer = Stemmer()
df["stems"] = df["tokens"].apply(lambda words: [stemmer.stem(w) for w in words])


In [ ]:
df["stems"][:5]

In [ ]:
df.head()

In [ ]:
# read stopwords
stopwords = pd.read_csv('Stopwords.csv', header=0)

In [ ]:
stopwords.head

In [ ]:
stops = set(stopwords['word'])


In [ ]:
df.info()

In [ ]:
df["clean_text1"] = df["stems"].apply(lambda word_list: [w for w in word_list if w not in stops])


In [ ]:
df.info()

In [ ]:
df["clean_text1"][44]

In [ ]:
df.head()

In [ ]:
df_final = df[["cat3_slug", "user_type", "clean_text1"]]

In [ ]:
df_final.head()

In [ ]:
df_final["clean_text"] = df_final["clean_text1"]

In [ ]:
df_final.head()

In [ ]:
df_final.drop(["clean_text1"], axis=1,inplace=True)

In [ ]:
df_final.head()

In [ ]:
df_final.to_csv("tehran_text_full_tamiz.csv", index=False)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from time import sleep

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
data_path = "/content/drive/MyDrive/Colab Notebooks/daqiqe/text_full_tamiz.csv"  # فایل CSV به عنوان مثال
df = pd.read_csv(data_path)
print(df.head())


In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 50  # You can adjust this based on text lengths

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df["clean_text"].astype(str))
sequences = tokenizer.texts_to_sequences(df["clean_text"].astype(str))
X_emb = pad_sequences(sequences, maxlen=max_len)

print("Embedding feature shape:", X_emb.shape)


۱. Sampling و تقسیم داده

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Sample 40% of the data
df_sampled = df.sample(frac=0.4, random_state=42)
texts = df_sampled["clean_text"].astype(str).tolist()
y = df_sampled["cat3_slug"]


۲. ویژگی‌سازی با BoW و TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

max_features = 10000

# Bag of Words
vectorizer_bow = CountVectorizer(max_features=max_features)
X_bow = vectorizer_bow.fit_transform(texts)

# TF-IDF
vectorizer_tfidf = TfidfVectorizer(max_features=max_features)
X_tfidf = vectorizer_tfidf.fit_transform(texts)


۳. تقسیم آموزش و تست (هر دو روش مشابه)


In [ ]:
# For BoW
X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

# For TF-IDF
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)


۴. مدل‌سازی با سه مدل (برای هر روش جداگانه)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_bow = LogisticRegression(max_iter=100)
lr_bow.fit(X_train_bow, y_train)
y_pred_bow = lr_bow.predict(X_test_bow)

lr_tfidf = LogisticRegression(max_iter=100)
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# استفاده از پارامترهای تنظیم‌شده برای افزایش سرعت
fast_rf_bow = RandomForestClassifier(
    n_estimators=20,      # تعداد کم درخت‌ها برای سرعت اولیه
    max_depth=10,         # محدود کردن عمق هر درخت
    n_jobs=-1,            # موازی‌سازی
    random_state=42,
    verbose=1             # نمایش روند اجرا (اختیاری)
)
fast_rf_bow.fit(X_train_bow, y_train)
y_pred_bow_rf_fast = fast_rf_bow.predict(X_test_bow)

fast_rf_tfidf = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
fast_rf_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf_rf_fast = fast_rf_tfidf.predict(X_test_tfidf)


**خط پاییت tuned هستش**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_bow = RandomForestClassifier(
    n_estimators=50,           # نه خیلی کم، نه خیلی زیاد
    max_depth=20,              # عمق متوسط، قابل تغییر
    min_samples_split=5,       # گره جدید با حداقل 5 نمونه
    min_samples_leaf=2,        # هر برگ حداقل 2 نمونه داشته باشد
    max_features='sqrt',       # برای تقسیم از جذر تعداد ویژگی‌ها استفاده کن
    class_weight='balanced',   # مقابله با کلاس‌های نامتوازن
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_bow.fit(X_train_bow, y_train)
y_pred_bow_rf_tuned = rf_bow.predict(X_test_bow)

rf_tfidf = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf_rf_tuned = rf_tfidf.predict(X_test_tfidf)


In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
y_pred_bow_nb = nb_bow.predict(X_test_bow)

nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf_nb = nb_tfidf.predict(X_test_tfidf)


۵. گزارش نتایج و مقایسه (هر دو روش)

In [ ]:
from sklearn.metrics import classification_report

print("=== Bag of Words (BoW) ===")
print("Logistic Regression:\n", classification_report(y_test, y_pred_bow))
print("Random Forest Fast:\n", classification_report(y_test, y_pred_bow_rf_fast))
print("Random Forest Tuned:\n", classification_report(y_test, y_pred_bow_rf_tuned))
print("Naive Bayes:\n", classification_report(y_test, y_pred_bow_nb))

print("\n=== TF-IDF ===")
print("Logistic Regression:\n", classification_report(y_test, y_pred_tfidf))
print("Random Forest Fast:\n", classification_report(y_test, y_pred_tfidf_rf_fast))
print("Random Forest Tuned:\n", classification_report(y_test, y_pred_tfidf_rf_tuned))
print("Naive Bayes:\n", classification_report(y_test, y_pred_tfidf_nb))


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def eval_model(y_true, y_pred, model_name, vec_name):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"{model_name} ({vec_name}) -> Accuracy: {acc:.3f}, Weighted F1: {f1:.3f}")

print("==== Simple results summary ====")
eval_model(y_test, y_pred_bow,             "LogReg",          "BoW")
eval_model(y_test, y_pred_bow_rf_fast,      "RandomForest",    "BoW")
eval_model(y_test, y_pred_tfidf_rf_tuned,    "RandomForest+",   "BoW")
eval_model(y_test, y_pred_bow_nb,          "NaiveBayes",      "BoW")

print("==========")
eval_model(y_test, y_pred_tfidf,           "LogReg",          "TF-IDF")
eval_model(y_test, y_pred_tfidf_rf_fast,    "RandomForest",    "TF-IDF")
eval_model(y_test, y_pred_tfidf_rf_tuned,  "RandomForest+",   "TF-IDF")
eval_model(y_test, y_pred_tfidf_nb,        "NaiveBayes",      "TF-IDF")


In [ ]:
!pip install gensim


In [ ]:
from gensim.models import Word2Vec

In [ ]:
# فرض: clean_text کاملاً پیش‌پردازش شده و هر رکورد یک جمله است
sentences = [str(text).split() for text in df_sampled["clean_text"]]


۳. آموزش مدل Word2Vec


In [ ]:
w2v_model = Word2Vec(
    sentences,
    vector_size=100,      # ابعاد بردارها (قابل تغییر)
    window=5,             # پنجره متن
    min_count=2,          # حداقل تکرار کلمه
    workers=4,            # تعداد هسته‌ها
    sg=1                  # استراتژی Skip-gram (دقیق‌تر در مسائل دسته‌بندی بزرگ)
)


ساخت بردار

In [ ]:
def sentence_vector(sentence, model):
    words = sentence.split()
    word_vecs = [model.wv[w] for w in words if w in model.wv]
    if len(word_vecs) == 0:
        return np.zeros(model.vector_size)
    else:
        return np.mean(word_vecs, axis=0)

X_w2v = np.array([sentence_vector(s, w2v_model) for s in df_sampled["clean_text"].astype(str)])


train & test

In [ ]:
from sklearn.model_selection import train_test_split

X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42
)

from sklearn.linear_model import LogisticRegression
logreg_w2v = LogisticRegression(max_iter=100)
logreg_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = logreg_w2v.predict(X_test_w2v)


In [ ]:
rf_w2v = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf_w2v.fit(X_train_w2v, y_train)
y_pred_rf_w2v = rf_w2v.predict(X_test_w2v)


In [ ]:
print("==== Simple results summary (Word2Vec) ====")
eval_model(y_test, y_pred_w2v,        "LogReg", "Word2Vec")
eval_model(y_test, y_pred_rf_w2v,     "RandomForest", "Word2Vec")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df_sampled["clean_text"].astype(str))

sequences = tokenizer.texts_to_sequences(df_sampled["clean_text"].astype(str))
X_seq = pad_sequences(sequences, maxlen=max_len)


from sklearn.preprocessing import LabelEncoder

y = df_sampled["cat3_slug"]
label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

num_classes = len(label_enc.classes_)

model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

print(model.summary())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_seq, y_encoded, test_size=0.2, random_state=42)

model.fit(X_train, y_train,
          epochs=10,
          batch_size=32,
          validation_data=(X_test, y_test))


قسمت سوال 3 که با
LSTM

پیش میبریم

In [ ]:

user_type_counts = df_sampled["user_type"].value_counts(dropna=False)
print(user_type_counts)

1. حذف رکوردهای Null و آماده‌سازی لیبل‌ها

In [ ]:

filtered_df = df_sampled[df_sampled["user_type"].notnull()]
texts = filtered_df["clean_text"].astype(str).tolist()
labels = filtered_df["user_type"].astype(str).tolist()

2. برچسب‌گذاری عددی


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(labels)


3. توکنایز و تبدیل به بردار عددی


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X_seq = pad_sequences(sequences, maxlen=max_len)


4. تقسیم آموزش/آزمون

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y, test_size=0.2, random_state=42, stratify=y
)


5. محاسبه class_weight (دقیق و بدون خطا)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("class_weight:", class_weight_dict)


6. ساخت و آموزش مدل LSTM با class_weight

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

num_classes = len(le.classes_)

model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=256,
    validation_data=(X_test, y_test),
    class_weight=class_weight_dict     # راهکار عدم توازن!
)


7. ارزیابی مدل و بررسی دقت هر کلاس

In [ ]:
from sklearn.metrics import classification_report

y_pred_lstm = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred_lstm, target_names=le.classes_))


8. پیش‌بینی user_type برای نمونه‌های Null


In [ ]:
unlabeled = df_sampled[df_sampled["user_type"].isnull()]
texts_unlabeled = unlabeled["clean_text"].astype(str).tolist()
X_seq_unlabeled = pad_sequences(tokenizer.texts_to_sequences(texts_unlabeled), maxlen=max_len)

pred_user_type = le.inverse_transform(np.argmax(model.predict(X_seq_unlabeled), axis=1))
df_sampled.loc[unlabeled.index, "predicted_user_type"] = pred_user_type


In [ ]:
import matplotlib.pyplot as plt

# شمارش فراوانی user_type پیش‌بینی شده
pred_counts = df_sampled.loc[df_sampled['predicted_user_type'].notnull(), 'predicted_user_type'].value_counts()

plt.figure(figsize=(6,4))
pred_counts.plot(kind='bar', color=['skyblue','coral'])
plt.title('Predicted user_type for unlabeled data')
plt.ylabel('Count')
plt.xlabel('user_type')
plt.show()


In [ ]:
pred_percent = pred_counts / pred_counts.sum() * 100
print(pred_percent)

In [ ]:
plt.hist(df_sampled['predicted_user_type'].dropna(), bins=2, color='orange', alpha=0.7)
plt.title('Distribution of predicted user_type')
plt.xlabel('user_type')
plt.ylabel('Frequency')
plt.show()


In [ ]:
counts = df_sampled.loc[df_sampled['predicted_user_type'].notnull(), 'predicted_user_type'].value_counts()
print(counts)


In [ ]:
import matplotlib.pyplot as plt

counts.plot(kind='bar', color=['skyblue','coral'])
plt.title('parakandegi user type bedone barchasb')
plt.xlabel('noe karbar')
plt.ylabel('tedad')
plt.show()
